In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Titanic: previsão de sobrevivência com Random Forest

Neste notebook eu treino um modelo de machine learning para prever quais passageiros do Titanic sobreviveram, a partir de dados como idade, sexo e classe da passagem. Passo por: exploração dos dados, limpeza, treino, validação e envio para a competição do Kaggle.

## 1. Carregando e conhecendo os dados

Leio o arquivo de treino com o pandas. A coluna `Survived` é o que queremos prever (1 = sobreviveu, 0 = não sobreviveu). As outras colunas são as pistas.

In [ ]:
treino = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
treino.head()

## 2. Verificando tamanho e valores vazios

Confiro quantos passageiros e colunas existem e onde há valores vazios, porque o modelo não aceita dados em branco.

In [ ]:
treino.shape

In [ ]:
treino.isnull().sum()


## 3. Limpando os dados

- Removi `Cabin` (quase 77% vazia) e `PassengerId`, `Name` e `Ticket` (só identificam o passageiro).
- Preenchi as idades vazias com a mediana e o porto de embarque vazio com o valor mais comum.

In [ ]:
treino = treino.drop(columns=["Cabin", "PassengerId", "Name", "Ticket"])


In [ ]:
treino["Age"] = treino["Age"].fillna(treino["Age"].median())
treino["Embarked"] = treino["Embarked"].fillna(treino["Embarked"].mode()[0])
treino.isnull().sum()

## 4. Transformando texto em números

O modelo só entende números. Converti `Sex` e `Embarked` em colunas de 0 e 1 com `get_dummies`.

In [ ]:
treino = pd.get_dummies(treino, columns=["Sex", "Embarked"], drop_first=True, dtype=int)
treino.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = treino.drop(columns=["Survived"])
y = treino["Survived"]

X_treino, X_validacao, y_treino, y_validacao = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_treino.shape, X_validacao.shape

## 6. Treinando o modelo e medindo a acurácia

Usei um Random Forest, que cria várias árvores de decisão e decide pelo voto delas. Na validação, o modelo acertou cerca de 82%.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_treino, y_treino)

In [ ]:
from sklearn.metrics import accuracy_score

palpites = modelo.predict(X_validacao)
accuracy_score(y_validacao, palpites)

## 7. Preparando os dados de teste

Apliquei a mesma limpeza ao arquivo de teste, para que ele tenha o mesmo formato dos dados de treino.

In [ ]:
teste = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
ids = teste["PassengerId"]
teste = teste.drop(columns=["Cabin", "PassengerId", "Name", "Ticket"])
teste["Age"] = teste["Age"].fillna(teste["Age"].median())
teste["Fare"] = teste["Fare"].fillna(teste["Fare"].median())
teste = pd.get_dummies(teste, columns=["Sex", "Embarked"], drop_first=True, dtype=int)
teste.isnull().sum()

## 8. Gerando as previsões e enviando para o Kaggle

O modelo prevê a sobrevivência de cada passageiro do teste e salva o resultado em `submission.csv`. No Kaggle, o score foi de cerca de 74,6%, abaixo da validação, porque o teste usa dados que o modelo nunca viu.

## Próximos passos

- Criar novas características a partir das colunas existentes (por exemplo, o tamanho da família).
- Testar outros modelos e ajustar os parâmetros do Random Forest.

In [ ]:
palpites_teste = modelo.predict(teste)

envio = pd.DataFrame({"PassengerId": ids, "Survived": palpites_teste})
envio.to_csv("submission.csv", index=False)
envio.head()

In [ ]:
envio.shape